In [1]:
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

In [2]:
model = joblib.load("../models/xgboost_fraud_pipeline.joblib")
preprocessor = joblib.load("../models/preprocessor.joblib")

In [3]:
with open("../models/threshold.json", "r") as file:
    threshold_config = json.load(file)

threshold = threshold_config["threshold"]

In [4]:
print("Threshold:", threshold)

Threshold: 0.8


In [5]:
from sklearn.model_selection import train_test_split

df = pd.read_csv("../Data/Processed/creditcard_clean.csv")

print("Cleaned dataset shape:")
print(df.shape)

Cleaned dataset shape:
(283726, 33)


In [6]:
print(df.columns.tolist())

['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class', 'Hour', 'TimeBin']


In [7]:
X = df.drop(columns=["Class", "Hour", "TimeBin"])
y = df["Class"]

X = X.copy()

X["Hour"] = (X["Time"] // 3600) % 24
X["Hour_sin"] = np.sin(2 * np.pi * X["Hour"] / 24)
X["Hour_cos"] = np.cos(2 * np.pi * X["Hour"] / 24)

X = X.drop(columns=["Hour"])

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("\nFeatures:")
print(X.columns.tolist())

Features shape: (283726, 32)
Target shape: (283726,)

Features:
['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Hour_sin', 'Hour_cos']


In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (226980, 32)
X_test shape: (56746, 32)
y_train shape: (226980,)
y_test shape: (56746,)


In [9]:
print("Training class distribution:")
print(y_train.value_counts().sort_index())

print("\nTest class distribution:")
print(y_test.value_counts().sort_index())

Training class distribution:
Class
0    226602
1       378
Name: count, dtype: int64

Test class distribution:
Class
0    56651
1       95
Name: count, dtype: int64


In [10]:
X_test_processed = preprocessor.transform(X_test)

print("Processed test shape:", X_test_processed.shape)

Processed test shape: (56746, 32)


In [11]:
y_prob = model.predict_proba(X_test_processed)[:, 1]

print("Probability shape:", y_prob.shape)
print("Minimum probability:", y_prob.min())
print("Maximum probability:", y_prob.max())

Probability shape: (56746,)
Minimum probability: 3.3772867e-09
Maximum probability: 0.999997


In [12]:
y_pred = (y_prob >= threshold).astype(int)

print("Threshold:", threshold)
print("Predicted class distribution:")
print(pd.Series(y_pred).value_counts().sort_index())

Threshold: 0.8
Predicted class distribution:
0    56668
1       78
Name: count, dtype: int64


In [13]:
import os

print(os.getcwd())

c:\Users\Administrator\Desktop\Fraud-Detection\notebooks


In [14]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

In [15]:
from src.models.evaluate import evaluate_model

print("Import successful")

Import successful


In [16]:
from src.models.evaluate import evaluate_model

metrics = evaluate_model(
    y_true=y_test,
    y_prob=y_prob,
    threshold=threshold
)

print("Production model metrics:")

for metric, value in metrics.items():
    print(f"{metric}: {value:.6f}")

Production model metrics:
precision: 0.935897
recall: 0.768421
f1: 0.843931
roc_auc: 0.972138
pr_auc: 0.813529


In [17]:
cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[56646     5]
 [   22    73]]


In [18]:
evaluation_summary = {
    "threshold": threshold,
    "precision": metrics["precision"],
    "recall": metrics["recall"],
    "f1": metrics["f1"],
    "roc_auc": metrics["roc_auc"],
    "pr_auc": metrics["pr_auc"],
    "true_negatives": cm[0, 0],
    "false_positives": cm[0, 1],
    "false_negatives": cm[1, 0],
    "true_positives": cm[1, 1],
}

evaluation_summary

{'threshold': 0.8,
 'precision': 0.9358974358974359,
 'recall': 0.7684210526315789,
 'f1': 0.8439306358381503,
 'roc_auc': 0.9721379935691198,
 'pr_auc': 0.8135294053352436,
 'true_negatives': np.int64(56646),
 'false_positives': np.int64(5),
 'false_negatives': np.int64(22),
 'true_positives': np.int64(73)}

In [19]:
y_prob_repeat = model.predict_proba(X_test_processed)[:, 1]

print(
    "Probabilities match:",
    np.allclose(y_prob, y_prob_repeat)
)

print(
    "Maximum probability difference:",
    np.max(np.abs(y_prob - y_prob_repeat))
)

Probabilities match: True
Maximum probability difference: 0.0


In [20]:
from src.data.ingestion import load_data
from src.data.cleaning import remove_duplicates
from src.data.validation import validate_target
from src.config import load_config
config = load_config("../config.yaml")

df = load_data("../"+config["paths"]["raw_data"])

clean = remove_duplicates(df)

validate_target(clean)

print("Target validation passed.")
print(clean["Class"].value_counts().sort_index())

Target validation passed.
Class
0    283253
1       473
Name: count, dtype: int64


In [21]:
X = clean.drop(columns=["Class"])
y = clean["Class"]

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nX columns:")
print(X.columns.tolist())

X shape: (283726, 30)
y shape: (283726,)

X columns:
['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount']


In [22]:
from src.features.engineering import create_time_features

X_engineered = create_time_features(X)

print("Engineered feature shape:", X_engineered.shape)
print("\nEngineered features:")
print(X_engineered.columns.tolist())

Engineered feature shape: (283726, 32)

Engineered features:
['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Hour_sin', 'Hour_cos']


In [23]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_engineered,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

print("\nTraining class distribution:")
print(y_train.value_counts().sort_index())

print("\nTest class distribution:")
print(y_test.value_counts().sort_index())

X_train: (226980, 32)
X_test: (56746, 32)
y_train: (226980,)
y_test: (56746,)

Training class distribution:
Class
0    226602
1       378
Name: count, dtype: int64

Test class distribution:
Class
0    56651
1       95
Name: count, dtype: int64


In [24]:
from src.preprocessing.preprocessing import (
    fit_preprocessor,
    transform_features
)

preprocessor = fit_preprocessor(X_train)

X_train_processed = transform_features(X_train, preprocessor)
X_test_processed = transform_features(X_test, preprocessor)

print("X_train processed:", X_train_processed.shape)
print("X_test processed:", X_test_processed.shape)



X_train processed: (226980, 32)
X_test processed: (56746, 32)


In [25]:
from src.preprocessing.preprocessing import save_preprocessor

save_preprocessor(
    preprocessor,
    "../models/preprocessor.joblib"
)

print("Preprocessor saved successfully.")

Preprocessor saved successfully.


In [26]:
from pathlib import Path

preprocessor_path = Path("../models/preprocessor.joblib")

print("Exists:", preprocessor_path.exists())
print("Size:", preprocessor_path.stat().st_size, "bytes")

Exists: True
Size: 4051 bytes


In [27]:
from src.models.train import train_model
from src.config import load_config

config = load_config("../config.yaml")

production_model = train_model(
    X_train_processed,
    y_train,
    config
)

print("Production training completed.")

Production training completed.


In [28]:
print("Pipeline type:", type(production_model).__name__)
print("Pipeline steps:", production_model.named_steps.keys())

print("\nModel parameters:")
print(production_model.named_steps["model"].get_params())

Pipeline type: Pipeline
Pipeline steps: dict_keys(['smote', 'model'])

Model parameters:
{'objective': 'binary:logistic', 'base_score': None, 'booster': None, 'callbacks': None, 'colsample_bylevel': None, 'colsample_bynode': None, 'colsample_bytree': 0.8, 'device': None, 'early_stopping_rounds': None, 'enable_categorical': False, 'eval_metric': None, 'feature_types': None, 'feature_weights': None, 'gamma': None, 'grow_policy': None, 'importance_type': None, 'interaction_constraints': None, 'learning_rate': 0.1, 'max_bin': None, 'max_cat_threshold': None, 'max_cat_to_onehot': None, 'max_delta_step': None, 'max_depth': 5, 'max_leaves': None, 'min_child_weight': None, 'missing': nan, 'monotone_constraints': None, 'multi_strategy': None, 'n_estimators': 300, 'n_jobs': None, 'num_parallel_tree': None, 'random_state': 42, 'reg_alpha': None, 'reg_lambda': None, 'sampling_method': None, 'scale_pos_weight': None, 'subsample': 0.8, 'tree_method': None, 'validate_parameters': None, 'verbosity': N

In [29]:
from src.models.train import save_model

save_model(
    production_model,
    "../models/xgboost_fraud_pipeline.joblib"
)

print("Production model saved successfully.")

Production model saved successfully.


In [30]:
from pathlib import Path

model_path = Path("../models/xgboost_fraud_pipeline.joblib")

print("Exists:", model_path.exists())
print("Size:", model_path.stat().st_size, "bytes")

Exists: True
Size: 750517 bytes


In [31]:
import json
import joblib

preprocessor = joblib.load("../models/preprocessor.joblib")
model = joblib.load("../models/xgboost_fraud_pipeline.joblib")

with open("../models/threshold.json", "r") as file:
    threshold = json.load(file)["threshold"]

print("Threshold:", threshold)

Threshold: 0.8


In [ ]:
from src.models.predict import predict_transaction

In [33]:
sample_transaction = X_test.iloc[[0]]

In [34]:
result = predict_transaction(
    sample_transaction,
    preprocessor,
    model,
    threshold
)

print(result)

{'fraud_probability': 2.2671161787002347e-05, 'prediction': 0}


In [35]:
import joblib

pipeline = joblib.load("../models/xgboost_fraud_pipeline.joblib")

print(type(pipeline))
print(pipeline)
print("\nSteps:")
print(pipeline.named_steps)

print("\nModel parameters:")
print(pipeline.named_steps["model"].get_params())

<class 'imblearn.pipeline.Pipeline'>
Pipeline(steps=[('smote', SMOTE(random_state=42)),
                ('model',
                 XGBClassifier(base_score=None, booster=None, callbacks=None,
                               colsample_bylevel=None, colsample_bynode=None,
                               colsample_bytree=0.8, device=None,
                               early_stopping_rounds=None,
                               enable_categorical=False, eval_metric=None,
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.1,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=5, max_leaves=None,
                               min_child_weight=None, mis

In [36]:
from api.schemas import TransactionRequest

print(TransactionRequest.model_fields.keys())
print("Number of input features:", len(TransactionRequest.model_fields))

dict_keys(['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount'])
Number of input features: 30


In [39]:
example = {
    "Time": 406,
    **{f"V{i}": 0.0 for i in range(1, 29)},
    "Amount": 10,
}

transaction = TransactionRequest(**example)

print(transaction.Time)
print(transaction.Amount)

406.0
10.0


In [41]:

import joblib
import json

preprocessor = joblib.load("../models/preprocessor.joblib")

print("Preprocessor type:", type(preprocessor))
print("\nInput features:")
print(preprocessor.feature_names_in_)

print("\nTransformers:")
print(preprocessor.transformers_)

print("\nOutput feature count:")
print(preprocessor.transform(pd.DataFrame(
    [[0.0] * len(preprocessor.feature_names_in_)],
    columns=preprocessor.feature_names_in_
)).shape)

Preprocessor type: <class 'sklearn.compose._column_transformer.ColumnTransformer'>

Input features:
['Time' 'V1' 'V2' 'V3' 'V4' 'V5' 'V6' 'V7' 'V8' 'V9' 'V10' 'V11' 'V12'
 'V13' 'V14' 'V15' 'V16' 'V17' 'V18' 'V19' 'V20' 'V21' 'V22' 'V23' 'V24'
 'V25' 'V26' 'V27' 'V28' 'Amount' 'Hour_sin' 'Hour_cos']

Transformers:
[('amount', Pipeline(steps=[('log',
                 FunctionTransformer(feature_names_out='one-to-one',
                                     func=<ufunc 'log1p'>)),
                ('scaler', StandardScaler())]), ['Amount']), ('numeric', StandardScaler(), ['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Hour_sin', 'Hour_cos'])]

Output feature count:
(1, 32)


In [42]:
with open("../models/threshold.json", "r") as f:
    threshold_info = json.load(f)

with open("../models/model_metadata.json", "r") as f:
    metadata = json.load(f)

print("Threshold information:")
print(threshold_info)

print("\nModel metadata:")
print(metadata)

Threshold information:
{'threshold': 0.8, 'selection_method': 'OOF F1 maximization', 'cv_strategy': '5-fold StratifiedKFold', 'random_state': 42}

Model metadata:
{'model': 'XGBoost', 'model_type': 'XGBClassifier', 'pipeline': ['SMOTE', 'XGBoost'], 'random_state': 42, 'hyperparameters': {'n_estimators': 300, 'max_depth': 5, 'learning_rate': 0.1, 'subsample': 0.8, 'colsample_bytree': 0.8}, 'tuning': {'method': 'RandomizedSearchCV', 'cv_folds': 3, 'scoring': 'average_precision', 'best_cv_pr_auc': 0.8501176962}, 'threshold_selection': {'method': 'OOF F1 maximization', 'cv_folds': 5, 'selected_threshold': 0.8, 'oof_pr_auc': 0.8437, 'oof_precision': 0.93865, 'oof_recall': 0.809524, 'oof_f1': 0.869318}, 'test_evaluation': {'threshold': 0.8, 'precision': 0.9359, 'recall': 0.7684, 'f1': 0.8439, 'roc_auc': 0.9721, 'pr_auc': 0.8135, 'true_negatives': 56646, 'false_positives': 5, 'false_negatives': 22, 'true_positives': 73}}


In [43]:
sample_transaction = X_test.iloc[0]

print(sample_transaction.to_dict())

{'Time': 61290.0, 'V1': 1.2288211502379, 'V2': -0.0634077165201056, 'V3': 0.274145142235826, 'V4': 0.647465021810117, 'V5': -0.0481345611508765, 'V6': 0.372073028593297, 'V7': -0.22423058741343, 'V8': 0.0799390492455152, 'V9': 0.640758817066441, 'V10': -0.273053702248503, 'V11': -1.25272793883718, 'V12': 0.465078770741453, 'V13': 0.400502115321077, 'V14': -0.292841860600363, 'V15': -0.10177401599731, 'V16': -0.399835897844616, 'V17': 0.0343356567914817, 'V18': -0.783550254934187, 'V19': 0.141344900433949, 'V20': -0.0965659023514416, 'V21': -0.129554448055005, 'V22': -0.0837793282428063, 'V23': -0.151661473916324, 'V24': -0.700371597289218, 'V25': 0.598550164523483, 'V26': 0.491409070563651, 'V27': 0.0029892597250263, 'V28': 0.0017822861144491, 'Amount': 11.5, 'Hour_sin': -0.9659258262890683, 'Hour_cos': -0.25881904510252063}


In [44]:
api_features = [
    "Time",
    *[f"V{i}" for i in range(1, 29)],
    "Amount"
]

print(sample_transaction[api_features].to_json())

{"Time":61290.0,"V1":1.2288211502,"V2":-0.0634077165,"V3":0.2741451422,"V4":0.6474650218,"V5":-0.0481345612,"V6":0.3720730286,"V7":-0.2242305874,"V8":0.0799390492,"V9":0.6407588171,"V10":-0.2730537022,"V11":-1.2527279388,"V12":0.4650787707,"V13":0.4005021153,"V14":-0.2928418606,"V15":-0.101774016,"V16":-0.3998358978,"V17":0.0343356568,"V18":-0.7835502549,"V19":0.1413449004,"V20":-0.0965659024,"V21":-0.1295544481,"V22":-0.0837793282,"V23":-0.1516614739,"V24":-0.7003715973,"V25":0.5985501645,"V26":0.4914090706,"V27":0.0029892597,"V28":0.0017822861,"Amount":11.5}
